## Context

This notebook accompanies User Guide section 8 (Authoring knowledge documents).  It walks through the exact thirty-line markdown shown in the guide, ingests it in regex mode, and inspects which constructs (heading, table row, bullet) yielded which triples and ENM entries.   It also calls out one prose sentence that does **not** survive regex extraction so the reader sees the rule concretely.

In [ ]:
# Phase 1B preamble: bootstrap runtime + load shared helpers
# Bootstrap: portable repo-root locator — walks up from CWD looking for
# tutorials/user_guide/notebooks/_shared.py. Works in both the source
# clone (knowlytix/GMS) and the public mirror clone (knowlytix/gmsh).
import sys
import warnings
from pathlib import Path
warnings.filterwarnings('ignore')

_here = Path.cwd().resolve()
for _candidate in [_here, *_here.parents]:
    if (_candidate / 'tutorials' / 'user_guide' / 'notebooks' / '_shared.py').exists():
        if str(_candidate) not in sys.path:
            sys.path.insert(0, str(_candidate))
        break

from tutorials.user_guide.notebooks import _helpers_phase1 as h1
h1.bootstrap_runtime()  # stubs modeva if absent
from tutorials.user_guide.notebooks import _shared as ug
import torch
DEVICE = ug.DEVICE
print(f'Device: {DEVICE}, CUDA available: {torch.cuda.is_available()}')


### Concept: write for the parser, not for the reader

The regex parser in `knowlytix.benchmark.ingest.ingest_markdown` extracts triples and ENM entries from three constructs: markdown headings (which set the section context), markdown tables (which yield one or more triples per row plus ENM entries for numeric cells), and bullet lists in `Key: Value` shape (one triple per line).  Free-text prose without entity mentions yields nothing in regex mode and only partial coverage in hybrid mode.  Authoring for ingest means preferring tables and bullets over paragraphs whenever the fact must be queryable.

Below we ingest a thirty-line synthetic markdown that mirrors the `data_science_practices.md` shape: an H1 title, an H2 with a recommendation table, an H3 with a numeric threshold table.  We then inspect the per-construct yield: how many triples come from the heading, how many from each table, how many ENM entries from the numeric cells.  A final cell appends a prose sentence (`RandomForest typically outperforms LogisticRegression`) and re-ingests; the triple count does not increase, demonstrating the rule that prose without entity mentions does not survive.

In [ ]:
import tempfile
import shutil
from pathlib import Path

from knowlytix.knowledge.config import DocGMSConfig
from knowlytix.knowledge.ingest import ingest_document
from knowlytix.knowledge.store import GMSExpertStore
from knowlytix.core.config import GeometryConfig, TrainConfig

SYNTHETIC_MD = (
    "# Iris Model Selection\n\n"
    "## Classification Rules\n\n"
    "| Data Profile | Recommended Model | Rationale |\n"
    "|---|---|---|\n"
    "| Need interpretability | LogisticRegression | "
    "Transparent coefficients |\n"
    "| Multiclass target | RandomForest | Native "
    "multiclass support |\n"
    "| Small dataset below 200 rows | SVM | Kernel "
    "trick effective |\n\n"
    "### Performance Thresholds\n\n"
    "| Risk Tier | Minimum Accuracy | Minimum F1 |\n"
    "|---|---|---|\n"
    "| Pilot | 0.85 | 0.80 |\n"
    "| Production | 0.90 | 0.85 |\n"
)
tmpdir = Path(tempfile.mkdtemp(prefix='ug_nb08_'))
src_path = tmpdir / 'iris_rules.md'
src_path.write_text(SYNTHETIC_MD)
print(f'Wrote {len(SYNTHETIC_MD)} chars to {src_path}')

# Count constructs by inspection (not by ingest)
lines = SYNTHETIC_MD.splitlines()
n_headings = sum(1 for l in lines if l.startswith('#'))
n_table_rows = sum(
    1 for l in lines
    if l.strip().startswith('|') and not l.strip().startswith('|--')
    and 'Data Profile' not in l
    and 'Risk Tier' not in l
)
n_bullets = sum(1 for l in lines if l.lstrip().startswith('- '))
print(f'Constructs: headings={n_headings}, '
      f'data table rows={n_table_rows}, '
      f'bullets={n_bullets}')


In [ ]:
# Ingest in regex mode
store_path = tmpdir / 'authored_store'
config = DocGMSConfig(
    store_path=str(store_path),
    geometry=GeometryConfig(m=64),
    train=TrainConfig(epochs=50, device=str(DEVICE)),
    ingest_mode='regex',
)
store = GMSExpertStore(config, device=DEVICE)
with ug.quiet():
    result = ingest_document(
        store, str(src_path), llm=None,
        config=config, device=DEVICE,
    )
n_triples = result.new_triples
n_enm = result.new_enm
print(f'Ingest: {n_triples} triples, {n_enm} ENM entries, '
      f'{result.new_entities} entities')


In [ ]:
# Inspect what was extracted
triples = list(store.doc_graph.triples)
print(f'First 10 triples (of {len(triples)}):')
for h, r, t in triples[:10]:
    print(f'  ({h}, {r}, {t})')

print(f'\nENM entries ({len(list(store.enm.keys()))}):')
for k in list(store.enm.keys()):
    val = store.enm.read_exact(k)
    val_f = float(val.item() if hasattr(val, 'item') else val)
    print(f'  [{k.type}::{k.id}] = {val_f}')


In [ ]:
# Bucket triples by construct (heuristic: based on relation name)
by_construct = {
    'heading-context': 0,
    'classification-table': 0,
    'threshold-table-numeric': 0,
    'other': 0,
}
for h, r, t in triples:
    rl = r.lower()
    if 'recommended_model' in rl or 'rationale' in rl:
        by_construct['classification-table'] += 1
    elif 'minimum_accuracy' in rl or 'minimum_f1' in rl:
        by_construct['threshold-table-numeric'] += 1
    elif 'has_section' in rl or 'is_part_of' in rl:
        by_construct['heading-context'] += 1
    else:
        by_construct['other'] += 1
for k, v in by_construct.items():
    print(f'  {k:30s} {v}')


In [ ]:
# Visualization: bar chart of triple yield per construct
import matplotlib.pyplot as plt

labels = list(by_construct.keys())
values = [by_construct[k] for k in labels]
fig, ax = plt.subplots(figsize=(8, 4.2))
ax.bar(labels, values, color='tab:blue')
for x, v in zip(labels, values):
    ax.text(x, v + max(values, default=1) * 0.02, str(v),
            ha='center', va='bottom', fontsize=10)
ax.set_ylabel('triple count')
ax.set_title('Regex-mode triple yield per markdown construct')
for label in ax.get_xticklabels():
    label.set_rotation(15)
    label.set_ha('right')
fig.tight_layout()
plt.show()


In [ ]:
# What does NOT survive: free-text prose
PROSE_ADDITION = (
    SYNTHETIC_MD
    + "\n## Notes\n\n"
    + "RandomForest typically outperforms "
    + "LogisticRegression on tabular data with mixed "
    + "types, especially when feature count exceeds "
    + "twenty.\n"
)
src_prose = tmpdir / 'iris_rules_prose.md'
src_prose.write_text(PROSE_ADDITION)

store_prose_path = tmpdir / 'prose_store'
config_prose = DocGMSConfig(
    store_path=str(store_prose_path),
    geometry=GeometryConfig(m=64),
    train=TrainConfig(epochs=50, device=str(DEVICE)),
    ingest_mode='regex',
)
store_prose = GMSExpertStore(config_prose, device=DEVICE)
with ug.quiet():
    result_prose = ingest_document(
        store_prose, str(src_prose), llm=None,
        config=config_prose, device=DEVICE,
    )
delta = result_prose.new_triples - n_triples
print(f'After adding prose paragraph: '
      f'{result_prose.new_triples} triples '
      f'(delta = {delta:+d}).  Regex extracts almost '
      f'nothing from free-text prose.')


### Exercises

1. Add the same fact as a bullet (`- RandomForest outperforms LogisticRegression`) and re-ingest.  Observe that the triple count now increases.
2. Verify the four numeric threshold cells round-trip exactly through ENM: `store.enm.read_exact(...)` should return 0.85, 0.80, 0.90, 0.85 byte-identically.
3. Deliberately misspell `LogisticRegression` as `Logistic Regression` (with a space) in one row and re-ingest.  Show that the parser now treats it as a different entity, illustrating why entity-name consistency matters.

### What you should now be able to do

- Author a markdown document whose triples ingest cleanly   in regex mode.
- Predict the triple yield from a heading, a table row,   and a bullet.
- Verify ENM round-trip on a numeric threshold cell.
- Recognise that free-text prose without entity mentions   does not survive regex extraction, and choose between   rewriting it as a table row, a bullet, or accepting   prose-only status.

### Cross-references

- USER_GUIDE.md section 8 (Authoring knowledge   documents)
- `knowlytix/harness/governance/knowledge/data_science_practices.md`   for the full reference document
- `knowlytix/benchmark/ingest.py:ingest_markdown` for the   table and bullet parsers

In [ ]:
# Final assertion: yield is at least heading + table-row count
assert n_triples >= n_headings + n_table_rows, (
    f'expected n_triples >= n_headings ({n_headings}) + '
    f'n_table_rows ({n_table_rows}); got {n_triples}'
)
# ENM should have at least one entry from the threshold cells
assert n_enm >= 1, (
    f'expected at least one ENM entry from the threshold '
    f'table; got {n_enm}'
)
print(f'OK: regex ingest yielded {n_triples} triples '
      f'and {n_enm} ENM entries from a 30-line markdown '
      f'(headings={n_headings}, '
      f'table rows={n_table_rows}).')
shutil.rmtree(tmpdir, ignore_errors=True)
